# Cleaning India PLFS microdata for GenAI labour-market analysis

This notebook prepares worker-level PLFS data for 2018–19, 2021–22, and 2023–24. It inspects the raw variables, selects and cleans the variables used in this project, adds readable labels, and saves one cleaned file per survey year.

The statistical analysis and figures are created separately in `scripts/run_analysis.py`.

In [2]:
%pip install pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


## 1. Setup

Import the libraries used throughout the notebook. `pandas` reads and transforms tabular data; `Path` creates file locations that work for any authorized user who follows this repository's folder structure.

In [3]:
import pandas as pd
from pathlib import Path

## 2. Explore the 2023–24 raw PLFS data

Begin with the newest PLFS person-level file. Reading a small preview first lets us inspect the raw column names and formats before loading the full dataset.

In [15]:
plfs_2023 = Path("../data/raw/plfs_2023_24_person.csv.gz")

preview = pd.read_csv(plfs_2023, nrows=5, dtype=str)
preview

,FI_perv1,B1q2_perv1,qtr_perv1,visit_perv1,b1q3_perv1,state_perv1,distcode_perv1,nss_region_perv1,b1q5_perv1,b1q6_perv1,...,b6q8_3pt7_perv1,b6q5_perv1,b6q6_perv1,b6q7_perv1,b6q9_perv1,b6q10_perv1,NSS_perv1,NSC_perv1,mult_perv1,no_qtr_perv1
0,FVP7,104,Q1,V1,1,02,04,021,01,14,...,0,94,NaN,NaN,0,0,2,4,246798,4
1,FVP7,104,Q1,V1,1,02,04,021,01,14,...,0,93,NaN,NaN,0,0,2,4,246798,4
2,FVP7,104,Q1,V1,1,02,04,021,01,14,...,0,11,01,611,0,6500,2,4,246798,4
3,FVP7,104,Q1,V1,1,02,04,021,01,14,...,3,21,01,611,0,0,2,4,246798,4
4,FVP7,104,Q1,V1,1,02,04,021,01,14,...,0,91,NaN,NaN,0,0,2,4,246798,4


In [16]:
preview.columns.tolist()

['FI_perv1',
 'B1q2_perv1',
 'qtr_perv1',
 'visit_perv1',
 'b1q3_perv1',
 'state_perv1',
 'distcode_perv1',
 'nss_region_perv1',
 'b1q5_perv1',
 'b1q6_perv1',
 'b1q11_perv1',
 'b1q12_perv1',
 'b1q1_perv1',
 'b1q13_perv1',
 'b1q14_perv1',
 'b1q15_perv1',
 'b4q1_perv1',
 'b4q4_perv1',
 'b4q5_perv1',
 'b4q6_perv1',
 'b4q7_perv1',
 'b4q8_perv1',
 'b4q9_perv1',
 'b4q10_perv1',
 'b4q11_perv1',
 'b4q12_perv1',
 'b4pt1q3_perv1',
 'b4pt1q4_perv1',
 'b4pt1q5_perv1',
 'b4pt1q6_perv1',
 'b4pt1q7_perv1',
 'b5pt1q3_perv1',
 'b5pt1q5_perv1',
 'b5pt1q6_perv1',
 'b5pt1q7_perv1',
 'b5pt1q8_perv1',
 'b5pt1q9_perv1',
 'b5pt1q10_perv1',
 'b5pt1q11_perv1',
 'b5pt1q12_perv1',
 'b5pt1q13_perv1',
 'b5pt1q14_perv1',
 'b5pt2q3_perv1',
 'b5pt2q5_perv1',
 'b5pt2q6_perv1',
 'b5pt2q7_perv1',
 'b5pt2q8_perv1',
 'b5pt2q9_perv1',
 'b5pt2q10_perv1',
 'b5pt2q11_perv1',
 'b5pt2q12_perv1',
 'b5pt2q13_perv1',
 'b5pt3q5_perv1',
 'b5pt3q6_perv1',
 'b5pt3q7_perv1',
 'b5pt3q8_perv1',
 'b5pt3q9_perv1',
 'b5pt3q10_perv1',
 'b5pt3

## 3. Select and rename the project variables

Load only the variables needed for this project: geography, demographic characteristics, activity status, industry, occupation, and the survey weight. The official PLFS variable names are then renamed to clear analysis-friendly names.

In [20]:
# specific columns selected from https://microdata.gov.in/NADA/index.php/catalog/213/data-dictionary/F4
important = [
    "state_perv1",
    "b1q3_perv1",
    "b4q5_perv1",
    "b4q6_perv1",
    "b4q8_perv1",
    "b5pt1q3_perv1",
    "b5pt1q5_perv1",
    "b5pt1q6_perv1",
    "mult_perv1"
]

preview[important].T

,0,1,2,3,4
state_perv1,02,02,02,02,02
b1q3_perv1,1,1,1,1,1
b4q5_perv1,1,2,1,2,1
b4q6_perv1,72,65,40,35,15
b4q8_perv1,08,06,08,08,07
b5pt1q3_perv1,94,93,11,21,91
b5pt1q5_perv1,NaN,NaN,01132,01132,NaN
b5pt1q6_perv1,NaN,NaN,611,611,NaN
mult_perv1,246798,246798,246798,246798,246798


In [21]:
data_plfs_2023 = pd.read_csv(
    plfs_2023,
    usecols=important,
    dtype=str
)

data_plfs_2023.shape

(418159, 9)

In [22]:
data_plfs_2023 = data_plfs_2023.rename(columns={
    "state_perv1": "state_code",
    "b1q3_perv1": "sector_code",
    "b4q5_perv1": "sex_code",
    "b4q6_perv1": "age",
    "b4q8_perv1": "education_code",
    "b5pt1q3_perv1": "usual_activity_status",
    "b5pt1q5_perv1": "industry_nic_code",
    "b5pt1q6_perv1": "occupation_nco_code",
    "mult_perv1": "survey_weight"
})

data_plfs_2023.head()

,sector_code,state_code,sex_code,age,education_code,usual_activity_status,industry_nic_code,occupation_nco_code,survey_weight
0,1,02,1,72,08,94,NaN,NaN,246798
1,1,02,2,65,06,93,NaN,NaN,246798
2,1,02,1,40,08,11,01132,611,246798
3,1,02,2,35,08,21,01132,611,246798
4,1,02,1,15,07,91,NaN,NaN,246798


In [25]:
data_plfs_2023["usual_activity_status"].value_counts(dropna=False).sort_index()

usual_activity_status
11     61281
12      6309
21     21491
31     46445
41       846
51     28151
81      8780
91    108977
92     59028
93     20604
94     18212
95      4041
97      4207
99     29787
Name: count, dtype: int64

## 4. Define the employed-worker population

The analysis focuses on people classified as employed under usual principal activity status. The six included codes cover self-employment, regular salaried/wage work, and casual wage work; people who are unemployed or outside the labour force are not included in the worker dataset.

In [26]:
# from page 36 of the pdf https://mospi.gov.in/sites/default/files/publication_reports/PLFS_changes_in_2025_Final.pdf
employed_codes = ["11", "12", "21", "31", "41", "51"]

workers_plfs_2023 = data_plfs_2023[
    data_plfs_2023["usual_activity_status"].isin(employed_codes)
].copy()

workers_plfs_2023.shape

(164523, 9)

In [28]:
workers_plfs_2023[
    "occupation_nco_code"
].notna()

2         True
3         True
7         True
9         True
10        True
          ... 
418146    True
418147    True
418150    True
418157    True
418158    True
Name: occupation_nco_code, Length: 164523, dtype: bool

## 5. Inspect codes and missing values

Review the values and formats of occupation, sex, age, and education. This confirms what the codes represent, identifies missing values, and checks why broad one-digit occupation groups are used for comparisons across all three survey years.

In [29]:
# how many employed workers have a non-missing occupation code
workers_plfs_2023["occupation_nco_code"].notna().sum()

np.int64(164523)

In [30]:
# how many employed workers have a missing occupation code
workers_plfs_2023["occupation_nco_code"].isna().sum()

np.int64(0)

In [32]:
workers_plfs_2023["occupation_nco_code"].str.len().value_counts(
    dropna=False
).sort_index()

occupation_nco_code
3    164523
Name: count, dtype: int64

In [34]:
# We use only the first digit in the main trend chart because:
# The 2018-19 PLFS uses an older occupation classification, NCO-2004.
# The later surveys use NCO-2015. Detailed codes may not mean exactly the same thing across the two versions.
# Broad one-digit groups are safer to compare across all three survey years.

workers_plfs_2023["occupation_nco_code"].str[0].value_counts(
    dropna=False
).sort_index()

occupation_nco_code
1     6669
2    10959
3     4917
4     5125
5    25329
6    48498
7    19895
8    11050
9    32081
Name: count, dtype: int64

In [35]:
workers_plfs_2023["sex_code"].value_counts(dropna=False)

sex_code
1    116235
2     48287
3         1
Name: count, dtype: int64

In [36]:
workers_plfs_2023["age"].describe()

count     164523
unique        81
top           40
freq        7803
Name: age, dtype: object

In [37]:
workers_plfs_2023["education_code"].value_counts(dropna=False).sort_index()

education_code
01    27502
02      160
03        2
04       67
05     8826
06    21597
07    38034
08    22295
10    18056
11     2972
12    18985
13     6027
Name: count, dtype: int64

In [39]:
# from page 33 of the pdf https://mospi.gov.in/sites/default/files/publication_reports/PLFS_changes_in_2025_Final.pdf
education_labels = {
    "01": "Not literate",
    "02": "Literate, no formal schooling",
    "03": "Literate, no formal schooling",
    "04": "Literate, no formal schooling",
    "05": "Below primary",
    "06": "Primary",
    "07": "Upper primary / middle",
    "08": "Secondary",
    "10": "Higher secondary",
    "11": "Diploma / certificate",
    "12": "Graduate",
    "13": "Postgraduate and above"
}

workers_plfs_2023["education_label"] = (
    workers_plfs_2023["education_code"]
    .map(education_labels)
)

workers_plfs_2023["education_label"].value_counts(dropna=False)

education_label
Upper primary / middle           38034
Not literate                     27502
Secondary                        22295
Primary                          21597
Graduate                         18985
Higher secondary                 18056
Below primary                     8826
Postgraduate and above            6027
Diploma / certificate             2972
Literate, no formal schooling      229
Name: count, dtype: int64

In [43]:
workers_plfs_2023.loc[
    workers_plfs_2023["sex_code"] == "3",
    ["sex_code", "age", "education_code", "occupation_nco_code", "survey_weight"]
]

,sex_code,age,education_code,occupation_nco_code,survey_weight
102672,3,38,12,753,2900669


In [44]:
workers_plfs_2023["age"] = pd.to_numeric(
    workers_plfs_2023["age"],
    errors="coerce"
)

workers_plfs_2023["age"].describe()

count    164523.000000
mean         40.369845
std          12.784213
min           8.000000
25%          30.000000
50%          40.000000
75%          50.000000
max          95.000000
Name: age, dtype: float64

In [46]:
#Some exact education codes have very few people—for example, codes 02, 03, and 04. 
# Combining categories gives you groups large enough to compare meaningfully later.
education_groups = {
    "01": "Below primary",
    "02": "Below primary",
    "03": "Below primary",
    "04": "Below primary",
    "05": "Below primary",
    "06": "Primary",
    "07": "Middle",
    "08": "Secondary",
    "10": "Higher secondary",
    "11": "Postsecondary",
    "12": "Postsecondary",
    "13": "Postsecondary"
}

workers_plfs_2023["education_group"] = (
    workers_plfs_2023["education_code"]
    .map(education_groups)
)

workers_plfs_2023["education_group"].value_counts()

education_group
Middle              38034
Below primary       36557
Postsecondary       27984
Secondary           22295
Primary             21597
Higher secondary    18056
Name: count, dtype: int64

In [47]:
workers_plfs_2023.info()

<class 'pandas.core.frame.DataFrame'>
Index: 164523 entries, 2 to 418158
Data columns (total 11 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   sector_code            164523 non-null  object
 1   state_code             164523 non-null  object
 2   sex_code               164523 non-null  object
 3   age                    164523 non-null  int64 
 4   education_code         164523 non-null  object
 5   usual_activity_status  164523 non-null  object
 6   industry_nic_code      164523 non-null  object
 7   occupation_nco_code    164523 non-null  object
 8   survey_weight          164523 non-null  object
 9   education_label        164523 non-null  object
 10  education_group        164523 non-null  object
dtypes: int64(1), object(10)
memory usage: 15.1+ MB


In [48]:
workers_plfs_2023[
    [
        "state_code",
        "sector_code",
        "sex_code",
        "age",
        "education_code",
        "usual_activity_status",
        "industry_nic_code",
        "occupation_nco_code",
        "survey_weight"
    ]
].sample(10, random_state=42)

,state_code,sector_code,sex_code,age,education_code,usual_activity_status,industry_nic_code,occupation_nco_code,survey_weight
269642,18,2,1,36,10,11,46306,522,693844
139881,09,1,1,32,10,51,41001,931,835518
126532,21,1,1,65,05,11,14109,112,3386111
147495,22,1,2,48,07,21,01124,611,5821633
213829,02,1,2,42,10,31,85101,234,545502
369051,10,1,1,26,12,21,01500,613,2995906
195856,19,2,1,25,07,11,43301,721,1527721
79152,19,2,1,42,07,12,49225,833,807843
374145,08,1,1,23,10,21,01500,613,2267396
257793,24,1,2,41,01,11,01412,612,2584512


## 6. Define readable labels for coded variables

PLFS records store many concepts as numeric codes. These dictionaries add readable labels while retaining the original codes, so the cleaned data remain transparent and can still be checked against official documentation.

In [49]:
# 1. State labels
state_labels = {
    "01": "Jammu and Kashmir",
    "02": "Himachal Pradesh",
    "03": "Punjab",
    "04": "Chandigarh",
    "05": "Uttarakhand",
    "06": "Haryana",
    "07": "Delhi",
    "08": "Rajasthan",
    "09": "Uttar Pradesh",
    "10": "Bihar",
    "11": "Sikkim",
    "12": "Arunachal Pradesh",
    "13": "Nagaland",
    "14": "Manipur",
    "15": "Mizoram",
    "16": "Tripura",
    "17": "Meghalaya",
    "18": "Assam",
    "19": "West Bengal",
    "20": "Jharkhand",
    "21": "Odisha",
    "22": "Chhattisgarh",
    "23": "Madhya Pradesh",
    "24": "Gujarat",
    "25": "Daman and Diu and Dadra and Nagar Haveli",
    "27": "Maharashtra",
    "28": "Andhra Pradesh",
    "29": "Karnataka",
    "30": "Goa",
    "31": "Lakshadweep",
    "32": "Kerala",
    "33": "Tamil Nadu",
    "34": "Puducherry",
    "35": "Andaman and Nicobar Islands",
    "36": "Telangana",
    "37": "Ladakh"
}

# 2. Rural / urban labels
sector_labels = {
    "1": "Rural",
    "2": "Urban"
}

# 3. Sex labels
sex_labels = {
    "1": "Male",
    "2": "Female",
    "3": "Transgender"
}

# 4. Official general-education labels
education_labels = {
    "01": "Not literate",
    "02": "Literate without formal schooling: EGS/NFEC/AEC",
    "03": "Literate without formal schooling: TLC",
    "04": "Literate without formal schooling: other",
    "05": "Below primary",
    "06": "Primary",
    "07": "Middle",
    "08": "Secondary",
    "10": "Higher secondary",
    "11": "Diploma/certificate course",
    "12": "Graduate",
    "13": "Postgraduate and above"
}

# 5. Usual principal activity-status labels
activity_labels = {
    "11": "Self-employed: own-account worker",
    "12": "Self-employed: employer",
    "21": "Self-employed: unpaid family worker",
    "31": "Regular salaried/wage employee",
    "41": "Casual wage labour: public works",
    "51": "Casual wage labour: other work",
    "81": "Unemployed: seeking/available for work",
    "91": "Attended educational institution",
    "92": "Domestic duties only",
    "93": "Domestic duties plus household-use production",
    "94": "Rentier/pensioner/remittance recipient",
    "95": "Unable to work due to disability",
    "97": "Other"
}

# 6. Broad occupation labels: first digit of the 3-digit NCO code
occupation_major_labels = {
    "1": "Managers",
    "2": "Professionals",
    "3": "Technicians and associate professionals",
    "4": "Clerical support workers",
    "5": "Service and sales workers",
    "6": "Skilled agricultural workers",
    "7": "Craft and related trades workers",
    "8": "Plant and machine operators",
    "9": "Elementary occupations"
}

# Create readable label columns without losing the original codes
mapped_workers_2023 = workers_plfs_2023.copy()

mapped_workers_2023["state_label"] = (
    mapped_workers_2023["state_code"].map(state_labels)
)

mapped_workers_2023["sector_label"] = (
    mapped_workers_2023["sector_code"].map(sector_labels)
)

mapped_workers_2023["sex_label"] = (
    mapped_workers_2023["sex_code"].map(sex_labels)
)

mapped_workers_2023["education_label"] = (
    mapped_workers_2023["education_code"].map(education_labels)
)

mapped_workers_2023["activity_label"] = (
    mapped_workers_2023["usual_activity_status"].map(activity_labels)
)

mapped_workers_2023["occupation_major_code"] = (
    mapped_workers_2023["occupation_nco_code"].str[0]
)

mapped_workers_2023["occupation_major_label"] = (
    mapped_workers_2023["occupation_major_code"]
    .map(occupation_major_labels)
)

In [50]:
mapped_workers_2023[
    [
        "state_label",
        "sector_label",
        "sex_label",
        "age",
        "education_label",
        "activity_label",
        "industry_nic_code",
        "occupation_nco_code",
        "occupation_major_label",
        "survey_weight"
    ]
].sample(10, random_state=42)

,state_label,sector_label,sex_label,age,education_label,activity_label,industry_nic_code,occupation_nco_code,occupation_major_label,survey_weight
269642,Assam,Urban,Male,36,Higher secondary,Self-employed: own-account worker,46306,522,Service and sales workers,693844
139881,Uttar Pradesh,Rural,Male,32,Higher secondary,Casual wage labour: other work,41001,931,Elementary occupations,835518
126532,Odisha,Rural,Male,65,Below primary,Self-employed: own-account worker,14109,112,Managers,3386111
147495,Chhattisgarh,Rural,Female,48,Middle,Self-employed: unpaid family worker,01124,611,Skilled agricultural workers,5821633
213829,Himachal Pradesh,Rural,Female,42,Higher secondary,Regular salaried/wage employee,85101,234,Professionals,545502
369051,Bihar,Rural,Male,26,Graduate,Self-employed: unpaid family worker,01500,613,Skilled agricultural workers,2995906
195856,West Bengal,Urban,Male,25,Middle,Self-employed: own-account worker,43301,721,Craft and related trades workers,1527721
79152,West Bengal,Urban,Male,42,Middle,Self-employed: employer,49225,833,Plant and machine operators,807843
374145,Rajasthan,Rural,Male,23,Higher secondary,Self-employed: unpaid family worker,01500,613,Skilled agricultural workers,2267396
257793,Gujarat,Rural,Female,41,Not literate,Self-employed: own-account worker,01412,612,Skilled agricultural workers,2584512


## 7. Clean the 2023–24 worker dataset

Standardize code formats, convert age and survey weight to numeric values, and retain employed people aged 15 or above with valid occupation codes and positive survey weights. Then add the readable labels defined above.

In [53]:
cleaned_2023 = data_plfs_2023.copy()

# Preserve the official code formats
cleaned_2023["state_code"] = cleaned_2023["state_code"].str.strip().str.zfill(2)
cleaned_2023["sector_code"] = cleaned_2023["sector_code"].str.strip()
cleaned_2023["sex_code"] = cleaned_2023["sex_code"].str.strip()
cleaned_2023["education_code"] = cleaned_2023["education_code"].str.strip().str.zfill(2)
cleaned_2023["usual_activity_status"] = (
    cleaned_2023["usual_activity_status"].str.strip().str.zfill(2)
)
cleaned_2023["industry_nic_code"] = (
    cleaned_2023["industry_nic_code"].str.strip().str.zfill(5)
)
cleaned_2023["occupation_nco_code"] = (
    cleaned_2023["occupation_nco_code"].str.strip().str.zfill(3)
)

# Convert real numbers from text to numeric values
cleaned_2023["age"] = pd.to_numeric(cleaned_2023["age"], errors="coerce")
cleaned_2023["survey_weight"] = pd.to_numeric(
    cleaned_2023["survey_weight"],
    errors="coerce"
)

# Keep employed people, age 15+, valid occupation codes, and usable weights
employed_codes = ["11", "12", "21", "31", "41", "51"]

cleaned_2023 = cleaned_2023.loc[
    cleaned_2023["usual_activity_status"].isin(employed_codes)
    & cleaned_2023["age"].ge(15)
    & cleaned_2023["survey_weight"].gt(0)
    & cleaned_2023["occupation_nco_code"].str[0].isin(list("123456789"))
].copy()

# Add readable labels using the dictionaries you already created
cleaned_2023["state_label"] = cleaned_2023["state_code"].map(state_labels)
cleaned_2023["sector_label"] = cleaned_2023["sector_code"].map(sector_labels)
cleaned_2023["sex_label"] = cleaned_2023["sex_code"].map(sex_labels)
cleaned_2023["education_label"] = cleaned_2023["education_code"].map(education_labels)
cleaned_2023["activity_label"] = (
    cleaned_2023["usual_activity_status"].map(activity_labels)
)

cleaned_2023["occupation_major_code"] = (
    cleaned_2023["occupation_nco_code"].str[0]
)

cleaned_2023["occupation_major_label"] = (
    cleaned_2023["occupation_major_code"].map(occupation_major_labels)
)

cleaned_2023.shape

(164419, 16)

In [54]:
sector_labels = {
    "1": "Rural",
    "2": "Urban"
}

sex_labels = {
    "1": "Male",
    "2": "Female",
    "3": "Transgender"
}

cleaned_2023["sector_label"] = (
    cleaned_2023["sector_code"].map(sector_labels)
)

cleaned_2023["sex_label"] = (
    cleaned_2023["sex_code"].map(sex_labels)
)

cleaned_2023[
    ["sector_code", "sector_label", "sex_code", "sex_label"]
].head()

,sector_code,sector_label,sex_code,sex_label
2,1,Rural,1,Male
3,1,Rural,2,Female
7,1,Rural,1,Male
9,1,Rural,1,Male
10,1,Rural,2,Female


In [55]:
cleaned_2023[
    ["sector_label", "sex_label"]
].isna().sum()

sector_label    0
sex_label       0
dtype: int64

In [56]:
education_labels = {
    "01": "Not literate",
    "02": "Literate without formal schooling: EGS/NFEC/AEC",
    "03": "Literate without formal schooling: TLC",
    "04": "Literate without formal schooling: other",
    "05": "Below primary",
    "06": "Primary",
    "07": "Middle",
    "08": "Secondary",
    "10": "Higher secondary",
    "11": "Diploma/certificate course",
    "12": "Graduate",
    "13": "Postgraduate and above"
}

cleaned_2023["education_label"] = (
    cleaned_2023["education_code"].map(education_labels)
)

cleaned_2023[
    ["education_code", "education_label"]
].value_counts().sort_index()

education_code  education_label                                
01              Not literate                                       27487
02              Literate without formal schooling: EGS/NFEC/AEC      160
03              Literate without formal schooling: TLC                 2
04              Literate without formal schooling: other              67
05              Below primary                                       8815
06              Primary                                            21552
07              Middle                                             38001
08              Secondary                                          22295
10              Higher secondary                                   18056
11              Diploma/certificate course                          2972
12              Graduate                                           18985
13              Postgraduate and above                              6027
Name: count, dtype: int64

In [57]:
cleaned_2023["education_label"].isna().sum()

np.int64(0)

In [59]:
activity_labels = {
    "11": "Self-employed: own-account worker",
    "12": "Self-employed: employer",
    "21": "Self-employed: unpaid family worker",
    "31": "Regular salaried/wage employee",
    "41": "Casual wage labour: public works",
    "51": "Casual wage labour: other work"
}

cleaned_2023["activity_label"] = (
    cleaned_2023["usual_activity_status"].map(activity_labels)
)

cleaned_2023[
    ["usual_activity_status", "activity_label"]
].value_counts().sort_index()

cleaned_2023["activity_label"].isna().sum()

np.int64(0)

In [60]:
state_labels = {
    "01": "Jammu and Kashmir",
    "02": "Himachal Pradesh",
    "03": "Punjab",
    "04": "Chandigarh",
    "05": "Uttarakhand",
    "06": "Haryana",
    "07": "Delhi",
    "08": "Rajasthan",
    "09": "Uttar Pradesh",
    "10": "Bihar",
    "11": "Sikkim",
    "12": "Arunachal Pradesh",
    "13": "Nagaland",
    "14": "Manipur",
    "15": "Mizoram",
    "16": "Tripura",
    "17": "Meghalaya",
    "18": "Assam",
    "19": "West Bengal",
    "20": "Jharkhand",
    "21": "Odisha",
    "22": "Chhattisgarh",
    "23": "Madhya Pradesh",
    "24": "Gujarat",
    "25": "Daman and Diu and Dadra and Nagar Haveli",
    "27": "Maharashtra",
    "28": "Andhra Pradesh",
    "29": "Karnataka",
    "30": "Goa",
    "31": "Lakshadweep",
    "32": "Kerala",
    "33": "Tamil Nadu",
    "34": "Puducherry",
    "35": "Andaman and Nicobar Islands",
    "36": "Telangana",
    "37": "Ladakh"
}

cleaned_2023["state_label"] = (
    cleaned_2023["state_code"].map(state_labels)
)

cleaned_2023[
    ["state_code", "state_label"]
].value_counts().sort_index()

state_code  state_label                             
01          Jammu and Kashmir                            4867
02          Himachal Pradesh                             3629
03          Punjab                                       5023
04          Chandigarh                                    506
05          Uttarakhand                                  2506
06          Haryana                                      3884
07          Delhi                                        1227
08          Rajasthan                                    7889
09          Uttar Pradesh                               13838
10          Bihar                                        6229
11          Sikkim                                       1469
12          Arunachal Pradesh                            3610
13          Nagaland                                     1889
14          Manipur                                      4165
15          Mizoram                                      2776
16          Tripu

In [61]:
cleaned_2023["state_label"].isna().sum()

np.int64(0)

In [62]:
occupation_major_labels = {
    "1": "Managers",
    "2": "Professionals",
    "3": "Technicians and associate professionals",
    "4": "Clerical support workers",
    "5": "Service and sales workers",
    "6": "Skilled agricultural workers",
    "7": "Craft and related trades workers",
    "8": "Plant and machine operators",
    "9": "Elementary occupations"
}

cleaned_2023["occupation_major_code"] = (
    cleaned_2023["occupation_nco_code"].str[0]
)

cleaned_2023["occupation_major_label"] = (
    cleaned_2023["occupation_major_code"]
    .map(occupation_major_labels)
)

cleaned_2023[
    [
        "occupation_nco_code",
        "occupation_major_code",
        "occupation_major_label"
    ]
].head(10)

,occupation_nco_code,occupation_major_code,occupation_major_label
2,611,6,Skilled agricultural workers
3,611,6,Skilled agricultural workers
7,611,6,Skilled agricultural workers
9,512,5,Service and sales workers
10,611,6,Skilled agricultural workers
12,611,6,Skilled agricultural workers
13,611,6,Skilled agricultural workers
16,263,2,Professionals
17,611,6,Skilled agricultural workers
19,931,9,Elementary occupations


In [63]:
cleaned_2023["occupation_major_label"].isna().sum()

np.int64(0)

## 8. Inspect the GenAI-score source table

This section documents the Gmyrek et al. task-level ISCO-08 score source and explains its columns. It does not add a GenAI score to the cleaned PLFS data; the occupation-to-GenAI exposure proxy is constructed later in the separate analysis script.

In [66]:
%pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 250 kB 2.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [68]:
genai_scores = pd.read_excel(
    "../data/raw/Final_Scores_ISCO08_Gmyrek_et_al_2025.xlsx"
)
genai_scores.head()

,label4d,label1d,ISCO_08,Title,taskID,Task_ISCO,score_2023,Weaviate Status,predicted_score_2025_gpt4o,prediction_justification_gpt4o,...,predicted_score_2025_gemini,prediction_justification_gemini,score_2025,source,mean_score_2023,mean_score_2025,SD_2023,SD_2025,potential25,potential23
0,1111 - Legislators,1 - Managers,1111,Legislators,1,Presiding over or participating in the proceed...,0.1,success,0.330,The task of presiding over or participating in...,...,0.35,The task of presiding over legislative proceed...,0.3400,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
1,1111 - Legislators,1 - Managers,1111,Legislators,2,"Determining, formulating and directing policie...",0.3,success,0.365,"The task of determining, formulating, and dire...",...,0.35,"The task of ""Determining, formulating and dire...",0.3575,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
2,1111 - Legislators,1 - Managers,1111,Legislators,3,"Making, ratifying, amending or repealing laws,...",0.2,success,0.330,"The task of making, ratifying, amending, or re...",...,0.25,"The task of making, ratifying, amending, or re...",0.2900,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
3,1111 - Legislators,1 - Managers,1111,Legislators,4,Serving on government administrative boards or...,0.2,success,0.220,"The task of ""Serving on government administrat...",...,0.28,The task of serving on government administrati...,0.2500,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected
4,1111 - Legislators,1 - Managers,1111,Legislators,5,Investigating matters of concern to the public...,0.3,success,0.340,"The task of ""Investigating matters of concern ...",...,0.35,"The task of ""Investigating matters of concern ...",0.3450,predicted,0.22,0.31,0.07,0.04,Not Exposed,Not affected


| Column | Meaning |
|---|---|
| `label4d` | Four-digit ISCO occupation code and occupation name together |
| `label1d` | Broad one-digit occupation group |
| `ISCO_08` | Four-digit ISCO-08 occupation code, such as `1111` |
| `Title` | Occupation name, such as “Legislators” |
| `taskID` | Identifier for a task within that occupation |
| `Task_ISCO` | Description of the task |
| `score_2025` | GenAI exposure score for that specific task |
| `mean_score_2025` | Average score for all tasks in that four-digit occupation |
| `potential25` | Source’s exposure category |

In [69]:
important_genai_columns = [
    "label4d",
    "label1d",
    "ISCO_08",
    "Title",
    "taskID",
    "Task_ISCO",
    "score_2025",
    "mean_score_2025",
    "potential25"
]

genai_scores[important_genai_columns].head(10)

,label4d,label1d,ISCO_08,Title,taskID,Task_ISCO,score_2025,mean_score_2025,potential25
0,1111 - Legislators,1 - Managers,1111,Legislators,1,Presiding over or participating in the proceed...,0.3400,0.31,Not Exposed
1,1111 - Legislators,1 - Managers,1111,Legislators,2,"Determining, formulating and directing policie...",0.3575,0.31,Not Exposed
2,1111 - Legislators,1 - Managers,1111,Legislators,3,"Making, ratifying, amending or repealing laws,...",0.2900,0.31,Not Exposed
3,1111 - Legislators,1 - Managers,1111,Legislators,4,Serving on government administrative boards or...,0.2500,0.31,Not Exposed
4,1111 - Legislators,1 - Managers,1111,Legislators,5,Investigating matters of concern to the public...,0.3450,0.31,Not Exposed
5,1111 - Legislators,1 - Managers,1111,Legislators,6,Attending community functions and meetings to ...,0.3350,0.31,Not Exposed
6,1111 - Legislators,1 - Managers,1111,Legislators,7,Negotiating with other legislators and represe...,0.2625,0.31,Not Exposed
7,1111 - Legislators,1 - Managers,1111,Legislators,8,"As members of the government, directing senior...",0.3250,0.31,Not Exposed
8,1112 - Senior Government Officials,1 - Managers,1112,Senior Government Officials,1,"Advising national, state, regional or local go...",0.3500,0.38,Minimal Exposure
9,1112 - Senior Government Officials,1 - Managers,1112,Senior Government Officials,2,Advising on the preparation of government budg...,0.4350,0.38,Minimal Exposure


In [70]:
genai_scores.loc[
    genai_scores["ISCO_08"] == 1111,
    [
        "Title",
        "taskID",
        "Task_ISCO",
        "score_2025",
        "mean_score_2025",
        "potential25"
    ]
]

,Title,taskID,Task_ISCO,score_2025,mean_score_2025,potential25
0,Legislators,1,Presiding over or participating in the proceed...,0.3400,0.31,Not Exposed
1,Legislators,2,"Determining, formulating and directing policie...",0.3575,0.31,Not Exposed
2,Legislators,3,"Making, ratifying, amending or repealing laws,...",0.2900,0.31,Not Exposed
3,Legislators,4,Serving on government administrative boards or...,0.2500,0.31,Not Exposed
4,Legislators,5,Investigating matters of concern to the public...,0.3450,0.31,Not Exposed
5,Legislators,6,Attending community functions and meetings to ...,0.3350,0.31,Not Exposed
6,Legislators,7,Negotiating with other legislators and represe...,0.2625,0.31,Not Exposed
7,Legislators,8,"As members of the government, directing senior...",0.3250,0.31,Not Exposed


In [71]:
genai_scores["score_2025"].describe()

count    3265.000000
mean        0.294074
std         0.165919
min         0.047500
25%         0.150000
50%         0.260000
75%         0.405000
max         0.762500
Name: score_2025, dtype: float64

In [74]:
print("Occupation lookup created:", "occupation_genai_lookup" in globals())
print("GenAI score added to clean data:", "genai_score" in cleaned_2023.columns)

Occupation lookup created: False
GenAI score added to clean data: False


## 9. Clean the 2021–22 worker dataset

Apply the same selection, format standardization, and label-mapping rules to the 2021–22 PLFS data. Using the same rules makes the resulting worker datasets comparable across years.

In [75]:
# Load only the PLFS columns needed for this project
columns_2021 = [
    "state_perv1",
    "b1q3_perv1",
    "b4q5_perv1",
    "b4q6_perv1",
    "b4q8_perv1",
    "b5pt1q3_perv1",
    "b5pt1q5_perv1",
    "b5pt1q6_perv1",
    "mult_perv1"
]

data_plfs_2021 = pd.read_csv(
    "../data/raw/plfs_2021_22_person.csv.gz",
    usecols=columns_2021,
    dtype=str,
    low_memory=False
)

# Give raw PLFS variables readable names
data_plfs_2021 = data_plfs_2021.rename(columns={
    "state_perv1": "state_code",
    "b1q3_perv1": "sector_code",
    "b4q5_perv1": "sex_code",
    "b4q6_perv1": "age",
    "b4q8_perv1": "education_code",
    "b5pt1q3_perv1": "usual_activity_status",
    "b5pt1q5_perv1": "industry_nic_code",
    "b5pt1q6_perv1": "occupation_nco_code",
    "mult_perv1": "survey_weight"
})

# Make a separate cleaned copy
cleaned_2021 = data_plfs_2021.copy()

# Preserve code formats, including leading zeroes
cleaned_2021["state_code"] = cleaned_2021["state_code"].str.strip().str.zfill(2)
cleaned_2021["education_code"] = (
    cleaned_2021["education_code"].str.strip().str.zfill(2)
)
cleaned_2021["usual_activity_status"] = (
    cleaned_2021["usual_activity_status"].str.strip().str.zfill(2)
)
cleaned_2021["industry_nic_code"] = (
    cleaned_2021["industry_nic_code"].str.strip().str.zfill(5)
)
cleaned_2021["occupation_nco_code"] = (
    cleaned_2021["occupation_nco_code"].str.strip().str.zfill(3)
)

# Convert true numeric variables
cleaned_2021["age"] = pd.to_numeric(cleaned_2021["age"], errors="coerce")
cleaned_2021["survey_weight"] = pd.to_numeric(
    cleaned_2021["survey_weight"],
    errors="coerce"
)

# Keep the project's worker population
employed_codes = ["11", "12", "21", "31", "41", "51"]

cleaned_2021 = cleaned_2021.loc[
    cleaned_2021["usual_activity_status"].isin(employed_codes)
    & cleaned_2021["age"].ge(15)
    & cleaned_2021["survey_weight"].gt(0)
    & cleaned_2021["occupation_nco_code"].str[0].isin(list("123456789"))
].copy()

# Add the same readable labels already created for 2023–24
cleaned_2021["state_label"] = cleaned_2021["state_code"].map(state_labels)
cleaned_2021["sector_label"] = cleaned_2021["sector_code"].map(sector_labels)
cleaned_2021["sex_label"] = cleaned_2021["sex_code"].map(sex_labels)
cleaned_2021["education_label"] = cleaned_2021["education_code"].map(education_labels)
cleaned_2021["activity_label"] = (
    cleaned_2021["usual_activity_status"].map(activity_labels)
)

cleaned_2021["occupation_major_code"] = (
    cleaned_2021["occupation_nco_code"].str[0]
)

cleaned_2021["occupation_major_label"] = (
    cleaned_2021["occupation_major_code"].map(occupation_major_labels)
)

cleaned_2021.shape

(157134, 16)

In [76]:
cleaned_2021[
    [
        "state_label",
        "sector_label",
        "sex_label",
        "education_label",
        "activity_label",
        "occupation_major_label"
    ]
].isna().sum()

state_label               0
sector_label              0
sex_label                 0
education_label           0
activity_label            0
occupation_major_label    0
dtype: int64

In [77]:
state_labels_2018 = state_labels.copy()

state_labels_2018["25"] = "Daman and Diu"
state_labels_2018["26"] = "Dadra and Nagar Haveli"

## 10. Clean the 2018–19 worker dataset

Apply the same worker-selection rules to 2018–19, while accounting for its older raw-variable suffix and separate Dadra and Nagar Haveli/Daman and Diu state codes. Its occupation classification is NCO-2004, so detailed occupation codes are not compared directly with later years.

In [78]:
columns_2018 = [
    "state_per_fv",
    "b1q3_per_fv",
    "b4q5_per_fv",
    "b4q6_per_fv",
    "b4q8_per_fv",
    "b5pt1q3_per_fv",
    "b5pt1q5_per_fv",
    "b5pt1q6_per_fv",
    "MULT_per_fv"
]

data_plfs_2018 = pd.read_csv(
    "../data/raw/plfs_2018_19_person.csv.gz",
    usecols=columns_2018,
    dtype=str,
    low_memory=False
)

data_plfs_2018 = data_plfs_2018.rename(columns={
    "state_per_fv": "state_code",
    "b1q3_per_fv": "sector_code",
    "b4q5_per_fv": "sex_code",
    "b4q6_per_fv": "age",
    "b4q8_per_fv": "education_code",
    "b5pt1q3_per_fv": "usual_activity_status",
    "b5pt1q5_per_fv": "industry_nic_code",
    "b5pt1q6_per_fv": "occupation_nco_code",
    "MULT_per_fv": "survey_weight"
})

cleaned_2018 = data_plfs_2018.copy()

# Preserve code formats
cleaned_2018["state_code"] = cleaned_2018["state_code"].str.strip().str.zfill(2)
cleaned_2018["education_code"] = (
    cleaned_2018["education_code"].str.strip().str.zfill(2)
)
cleaned_2018["usual_activity_status"] = (
    cleaned_2018["usual_activity_status"].str.strip().str.zfill(2)
)
cleaned_2018["industry_nic_code"] = (
    cleaned_2018["industry_nic_code"].str.strip().str.zfill(5)
)
cleaned_2018["occupation_nco_code"] = (
    cleaned_2018["occupation_nco_code"].str.strip().str.zfill(3)
)

# Convert numeric variables
cleaned_2018["age"] = pd.to_numeric(cleaned_2018["age"], errors="coerce")
cleaned_2018["survey_weight"] = pd.to_numeric(
    cleaned_2018["survey_weight"],
    errors="coerce"
)

# Keep the same project population as the later years
cleaned_2018 = cleaned_2018.loc[
    cleaned_2018["usual_activity_status"].isin(employed_codes)
    & cleaned_2018["age"].ge(15)
    & cleaned_2018["survey_weight"].gt(0)
    & cleaned_2018["occupation_nco_code"].str[0].isin(list("123456789"))
].copy()

# Add labels
cleaned_2018["state_label"] = (
    cleaned_2018["state_code"].map(state_labels_2018)
)
cleaned_2018["sector_label"] = cleaned_2018["sector_code"].map(sector_labels)
cleaned_2018["sex_label"] = cleaned_2018["sex_code"].map(sex_labels)
cleaned_2018["education_label"] = (
    cleaned_2018["education_code"].map(education_labels)
)
cleaned_2018["activity_label"] = (
    cleaned_2018["usual_activity_status"].map(activity_labels)
)

cleaned_2018["occupation_major_code"] = (
    cleaned_2018["occupation_nco_code"].str[0]
)

cleaned_2018["occupation_major_label"] = (
    cleaned_2018["occupation_major_code"].map(occupation_major_labels)
)

cleaned_2018.shape

(143183, 16)

In [79]:
cleaned_2018[
    [
        "state_label",
        "sector_label",
        "sex_label",
        "education_label",
        "activity_label",
        "occupation_major_label"
    ]
].isna().sum()

state_label                0
sector_label               0
sex_label                  0
education_label           54
activity_label             0
occupation_major_label     0
dtype: int64

In [80]:
cleaned_2018["education_label"] = (
    cleaned_2018["education_code"]
    .map(education_labels)
    .fillna("Missing / not reported")
)

In [81]:
# Original raw education information is genuinely missing for 54 people
cleaned_2018["education_code"].isna().sum()

np.int64(54)

In [82]:
# Every row now has a readable education label
cleaned_2018["education_label"].isna().sum()

np.int64(0)

## 11. Add survey-year metadata and save the cleaned files

Add the survey year and occupation-classification version, reset row indexes, and save one compressed cleaned file per year in `data/cleaned/`. These files are the only PLFS inputs used by the separate analysis script.

In [83]:
cleaned_2018["year"] = "2018-19"
cleaned_2018["occupation_classification"] = "NCO-2004"

cleaned_2021["year"] = "2021-22"
cleaned_2021["occupation_classification"] = "NCO-2015"

cleaned_2023["year"] = "2023-24"
cleaned_2023["occupation_classification"] = "NCO-2015"

In [84]:
cleaned_2018[["year", "occupation_classification"]].head()

,year,occupation_classification
0,2018-19,NCO-2004
2,2018-19,NCO-2004
9,2018-19,NCO-2004
14,2018-19,NCO-2004
18,2018-19,NCO-2004


In [86]:
cleaned_2018 = cleaned_2018.reset_index(drop=True)
cleaned_2021 = cleaned_2021.reset_index(drop=True)
cleaned_2023 = cleaned_2023.reset_index(drop=True)

In [87]:
cleaned_2018[["year", "occupation_classification"]].head()

,year,occupation_classification
0,2018-19,NCO-2004
1,2018-19,NCO-2004
2,2018-19,NCO-2004
3,2018-19,NCO-2004
4,2018-19,NCO-2004


In [88]:
from pathlib import Path

cleaned_folder = Path("../data/cleaned")
cleaned_folder.mkdir(exist_ok=True)

cleaned_2018.to_csv(
    cleaned_folder / "plfs_2018_19_workers_clean.csv.gz",
    index=False,
    compression="gzip"
)

cleaned_2021.to_csv(
    cleaned_folder / "plfs_2021_22_workers_clean.csv.gz",
    index=False,
    compression="gzip"
)

cleaned_2023.to_csv(
    cleaned_folder / "plfs_2023_24_workers_clean.csv.gz",
    index=False,
    compression="gzip"
)

In [89]:
for file_path in sorted(cleaned_folder.glob("*.csv.gz")):
    print(file_path.name, f"{file_path.stat().st_size / 1_000_000:.1f} MB")

plfs_2018_19_workers_clean.csv.gz 1.7 MB
plfs_2021_22_workers_clean.csv.gz 2.1 MB
plfs_2023_24_workers_clean.csv.gz 2.2 MB
